# ESN + Ridge Regression Baseline

Ziel: Mackey-Glass-Zeitreihe (tau=17) mit einem Echo State Network
und Ridge Regression als Readout vorhersagen.

**Befund:** alpha=1e-6 fuehrt zu katastrophalem Overfitting (NRMSE > 1).
Dieses Notebook dokumentiert den Effekt und vergleicht drei alpha-Werte.

In [1]:
import matplotlib
matplotlib.use('Agg')
from src.data.mackey_glass import generate_mackey_glass, prepare_dataset
from src.reservoir.esn import EchoStateNetwork
from src.baselines.ridge_readout import RidgeReadout
import numpy as np
import matplotlib.pyplot as plt
import time

In [2]:
# Mackey-Glass Zeitreihe
daten = generate_mackey_glass(n_steps=12000, tau=17)
X_train, y_train, X_test, y_test = prepare_dataset(daten, train_ratio=0.8)

print(f"Trainings-Samples: {len(X_train)}")
print(f"Test-Samples:      {len(X_test)}")

plt.figure(figsize=(12, 3))
plt.plot(daten[:500])
plt.title("Mackey-Glass Zeitreihe (erste 500 Schritte)")
plt.xlabel("Zeitschritt")
plt.ylabel("x(t)")
plt.tight_layout()
plt.savefig('mackey_glass_preview.png', dpi=80)
plt.close()
print("Plot gespeichert: mackey_glass_preview.png")

Trainings-Samples: 9599
Test-Samples:      2400
Plot gespeichert: mackey_glass_preview.png


In [3]:
# ESN einmalig aufbauen und Reservoir-Zustaende berechnen
esn = EchoStateNetwork(n_reservoir=500, spectral_radius=0.9, sparsity=0.1, seed=42)

t0 = time.time()
train_states = esn.run(X_train)
t_reservoir = time.time() - t0

# Warmup: erste 100 Zustaende verwerfen
warmup = 100
train_states_cut = train_states[warmup:]
y_train_cut = y_train[warmup:]

# Test-Zustaende
esn.reset()
test_states = esn.run(X_test)

print(f"Reservoir-Durchlauf: {t_reservoir:.2f}s")
print(f"Trainings-Zustaende (nach Warmup): {train_states_cut.shape}")

Reservoir-Durchlauf: 0.33s
Trainings-Zustaende (nach Warmup): (9499, 500)


In [4]:
# Alpha-Vergleich: drei Regularisierungsstaerken testen
alphas = [1e-6, 1e-4, 1e-2]
results = {}

for alpha in alphas:
    readout = RidgeReadout(alpha=alpha)
    t0 = time.time()
    readout.fit(train_states_cut, y_train_cut)
    t_fit = time.time() - t0
    
    nrmse_train = readout.score(train_states_cut, y_train_cut)
    nrmse_test = readout.score(test_states, y_test)
    y_pred = readout.predict(test_states)
    
    results[alpha] = {
        'nrmse_train': nrmse_train,
        'nrmse_test': nrmse_test,
        'y_pred': y_pred,
        't_fit_ms': t_fit * 1000
    }
    print(f"alpha={alpha:.0e}: Train-NRMSE={nrmse_train:.4f}, Test-NRMSE={nrmse_test:.4f}, Fit={t_fit*1000:.1f}ms")

alpha=1e-06: Train-NRMSE=0.0007, Test-NRMSE=1.1475, Fit=24.1ms


alpha=1e-04: Train-NRMSE=0.0015, Test-NRMSE=0.5496, Fit=186.4ms


alpha=1e-02: Train-NRMSE=0.0040, Test-NRMSE=0.2735, Fit=183.3ms


In [5]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9))
n_plot = 300

for i, alpha in enumerate(alphas):
    y_pred = results[alpha]['y_pred']
    nrmse = results[alpha]['nrmse_test']
    axes[i].plot(y_test[:n_plot], label="Tatsaechlich", linewidth=1.5)
    axes[i].plot(y_pred[:n_plot], label="Vorhersage", linewidth=1.5, linestyle="--", alpha=0.8)
    axes[i].set_title(f"alpha={alpha:.0e} -- Test-NRMSE={nrmse:.4f}")
    axes[i].set_ylabel("x(t)")
    axes[i].legend(loc='upper right', fontsize=8)

axes[-1].set_xlabel("Zeitschritt")
plt.suptitle("ESN + Ridge Regression: Alpha-Vergleich", fontsize=13)
plt.tight_layout()
plt.savefig('alpha_comparison.png', dpi=80)
plt.close()
print("Plot gespeichert: alpha_comparison.png")

Plot gespeichert: alpha_comparison.png


## Ergebnisse: Alpha-Vergleich

| alpha | Train-NRMSE | Test-NRMSE | Bewertung |
|-------|-------------|------------|-----------|
| 1e-6  | 0.0007 | 1.1475 | Katastrophales Overfitting |
| 1e-4  | 0.0015 | 0.5496 | Noch stark ueberangepasst |
| 1e-2  | 0.0040 | 0.2735 | Besser, aber noch > 0.1 |

**Ziel (NRMSE < 0.1) wird erst bei alpha ~ 10 erreicht (NRMSE=0.09).**

## Lernbefund: Ridge-Regularisierung und Ill-Conditioned Gram-Matrix

Die Gram-Matrix X^T X des Reservoirs hat Konditionszahl ~2e11.
Alpha = 1e-6 ist gegenueber Eigenwerten von bis zu 2e5 voellig unsichtbar --
die Regularisierung hat keinen praktischen Effekt und das Modell memoriert die Trainingsdaten.

**Faustregel fuer ESN + Ridge auf Mackey-Glass:** alpha >= 1 (typisch 10-100).
Kleine alpha (1e-6 bis 1e-2) fuehren zu Overfitting bei 9500 Trainingspunkten
und 500 Reservoir-Neuronen.

## Naechster Schritt
HDC-Readout als Alternative zu Ridge Regression (Auftrag 02).